# 06 - Postprocessing: Decoding, Detokenization & Filtering

---

## Learning Objectives

By the end of this notebook you will be able to:

- Perform **detokenization**: convert token IDs back to readable text, merge subwords, remove special tokens.
- Implement **greedy decoding** and explain its limitations.
- Implement **beam search** and understand the trade-off between beam width and quality.
- Explain and demonstrate **temperature scaling** and its effect on output distributions.
- Describe **top-k** and **top-p (nucleus)** sampling strategies.
- Apply **output filtering**: repetition removal, length constraints, and safety checks.

## Prerequisites

- Completed Notebooks 01-05 (text preprocessing, tokenization, evaluation).
- Basic probability (softmax, sampling).
- Understanding of language model prediction (next-token probabilities).

## Table of Contents

1. [Post-Text Processing: Why It Matters](#1-post-text-processing-why-it-matters)
2. [Detokenization](#2-detokenization)
3. [Decoding Strategies Overview](#3-decoding-strategies-overview)
4. [Greedy Decoding](#4-greedy-decoding)
5. [Beam Search](#5-beam-search)
6. [Temperature Scaling](#6-temperature-scaling)
7. [Top-k and Top-p Sampling (Preview)](#7-top-k-and-top-p-sampling-preview)
8. [Output Filtering](#8-output-filtering)
9. [Common Mistakes & Debugging Tips](#9-common-mistakes--debugging-tips)
10. [Exercises](#10-exercises)
11. [Exercise Solutions](#11-exercise-solutions)

In [ ]:
# ---- Environment setup ----
import sys
sys.path.insert(0, "../..")
from src.utils.seed import set_global_seed

set_global_seed(42)

import numpy as np
import math
from collections import Counter
from typing import List, Dict, Tuple, Optional

print("Setup complete.")

## 1. Post-Text Processing: Why It Matters

Notebook 01 covered **pre-processing** (cleaning raw text before feeding it to a model). This notebook covers the other end: **post-processing** (transforming model outputs into usable text).

```
Raw Text  -->  [Pre-processing]  -->  Token IDs  -->  [MODEL]  -->  Logits/Probs
                                                                        |
                                                                        v
Final Text  <--  [Post-processing]  <--  Token IDs  <--  [Decoding Strategy]
```

Post-processing includes:
- **Decoding**: choosing which tokens to output from probability distributions.
- **Detokenization**: converting token IDs back into readable text.
- **Filtering**: removing repetition, enforcing length constraints, safety checks.

## 2. Detokenization

### The Problem

Models output sequences of **token IDs**. We need to convert these back to human-readable text:

1. **Map IDs to tokens**: Look up each ID in the vocabulary.
2. **Merge subwords**: `["transform", "##er"]` -> `"transformer"` (WordPiece) or `["trans", "former"]` -> `"transformer"` (BPE).
3. **Remove special tokens**: `[CLS]`, `[SEP]`, `<s>`, `</s>`, `<pad>`.
4. **Fix spacing and punctuation**: `"Hello , world !"` -> `"Hello, world!"`.

In [ ]:
# ---- Detokenization implementations ----

class SimpleDetokenizer:
    """Detokenizer that handles subword merging and special token removal."""
    
    SPECIAL_TOKENS = {"[CLS]", "[SEP]", "[PAD]", "[MASK]", "[UNK]",
                      "<s>", "</s>", "<pad>", "<eos>", "<bos>"}
    
    @staticmethod
    def detokenize_wordpiece(tokens: List[str]) -> str:
        """Detokenize WordPiece tokens (## prefix for continuations)."""
        # Remove special tokens
        tokens = [t for t in tokens if t not in SimpleDetokenizer.SPECIAL_TOKENS]
        
        if not tokens:
            return ""
        
        result = tokens[0]
        for token in tokens[1:]:
            if token.startswith("##"):
                # Continuation token: merge without space
                result += token[2:]
            else:
                # New word: add space
                result += " " + token
        
        return result
    
    @staticmethod
    def detokenize_bpe(tokens: List[str], eow_marker: str = "</w>") -> str:
        """Detokenize BPE tokens (end-of-word marker)."""
        tokens = [t for t in tokens if t not in SimpleDetokenizer.SPECIAL_TOKENS]
        
        result = ""
        for token in tokens:
            if token.endswith(eow_marker):
                result += token[:-len(eow_marker)] + " "
            else:
                result += token
        
        return result.strip()
    
    @staticmethod
    def fix_punctuation_spacing(text: str) -> str:
        """Fix spacing around punctuation marks."""
        import re
        # Remove space before punctuation
        text = re.sub(r"\s+([.,!?;:'\")])", r"\1", text)
        # Remove space after opening brackets/quotes
        text = re.sub(r"([\(\"'])\s+", r"\1", text)
        # Collapse multiple spaces
        text = re.sub(r"\s+", " ", text).strip()
        return text


detok = SimpleDetokenizer()

# WordPiece example
wp_tokens = ["[CLS]", "the", "transform", "##er", "model", "is",
             "un", "##believ", "##ably", "power", "##ful", ".", "[SEP]"]

print("WordPiece Detokenization:")
print(f"  Tokens:  {wp_tokens}")
print(f"  Text:    {detok.detokenize_wordpiece(wp_tokens)}")
print(f"  Fixed:   {detok.fix_punctuation_spacing(detok.detokenize_wordpiece(wp_tokens))}")
print()

# BPE example
bpe_tokens = ["<s>", "the</w>", "trans", "form", "er</w>", "mod", "el</w>",
              "is</w>", "great</w>", ".</w>", "</s>"]

print("BPE Detokenization:")
print(f"  Tokens:  {bpe_tokens}")
print(f"  Text:    {detok.detokenize_bpe(bpe_tokens)}")

In [ ]:
# ---- Token ID to text pipeline ----

# Simulated vocabulary (ID -> token)
id_to_token = {
    0: "<pad>", 1: "<s>", 2: "</s>", 3: "<unk>",
    4: "the", 5: "cat", 6: "sat", 7: "on", 8: "mat",
    9: "a", 10: "dog", 11: "is", 12: "big", 13: ".",
    14: "run", 15: "##ning", 16: "##s", 17: "happy",
    18: "##ness", 19: "un", 20: "##happy",
}

def ids_to_text(token_ids: List[int], vocab: Dict[int, str]) -> str:
    """Convert token IDs to detokenized text."""
    tokens = [vocab.get(tid, "<unk>") for tid in token_ids]
    text = detok.detokenize_wordpiece(tokens)
    text = detok.fix_punctuation_spacing(text)
    return text

# Example
token_ids = [1, 4, 12, 5, 11, 14, 15, 13, 2, 0, 0]
print(f"Token IDs: {token_ids}")
print(f"Tokens:    {[id_to_token.get(i, '?') for i in token_ids]}")
print(f"Text:      {ids_to_text(token_ids, id_to_token)}")

## 3. Decoding Strategies Overview

At each step, a language model outputs a probability distribution over the vocabulary. **Decoding** decides which token to select.

| Strategy | Method | Quality | Diversity | Speed |
|----------|--------|:-------:|:---------:|:-----:|
| **Greedy** | Pick argmax at each step | Low-medium | None | Very fast |
| **Beam Search** | Track top-$k$ sequences | Medium-high | Low | Medium |
| **Sampling** | Sample from distribution | Variable | High | Fast |
| **Top-k Sampling** | Sample from top $k$ tokens | Good | Medium-high | Fast |
| **Top-p (Nucleus)** | Sample from smallest set summing to $p$ | Good | Medium-high | Fast |
| **Temperature + Sampling** | Scale logits, then sample | Tunable | Tunable | Fast |

## 4. Greedy Decoding

At each step, pick the token with the **highest probability**:

$$w_t = \arg\max_{w} P(w | w_{1}, \ldots, w_{t-1})$$

### Limitations

- **Locally optimal, not globally optimal**: The best token at each step may not lead to the best overall sequence.
- **No diversity**: Always produces the same output for the same input.
- **Repetition prone**: Can get stuck in loops.

In [ ]:
# ---- Implement greedy decoding ----

class SimpleLanguageModel:
    """A toy language model using bigram probabilities for demonstration.
    
    In real models, these probabilities would come from a neural network.
    Here we define them manually to illustrate decoding strategies.
    """
    
    def __init__(self):
        self.vocab = ["<s>", "</s>", "the", "cat", "dog", "sat", "on",
                      "mat", "a", "big", "small", "red", "ran", "quickly",
                      "slowly", "and", "chased"]
        self.token_to_id = {t: i for i, t in enumerate(self.vocab)}
        self.vocab_size = len(self.vocab)
        
        # Define transition probabilities (simplified bigram)
        # P(next_token | current_token) as logits
        np.random.seed(42)
        self._logits = np.random.randn(self.vocab_size, self.vocab_size) * 0.5
        
        # Set some meaningful transitions
        self._set_transition("<s>", {"the": 3.0, "a": 2.5})
        self._set_transition("the", {"cat": 2.5, "dog": 2.0, "big": 1.5, "small": 1.0, "red": 1.0, "mat": 1.5})
        self._set_transition("a", {"cat": 2.0, "dog": 2.0, "big": 2.5, "small": 2.0, "red": 1.5})
        self._set_transition("cat", {"sat": 2.5, "ran": 2.0, "chased": 1.5, "and": 1.0})
        self._set_transition("dog", {"sat": 2.0, "ran": 2.5, "chased": 2.0, "and": 1.0})
        self._set_transition("sat", {"on": 3.0, "quickly": 1.0, "slowly": 1.0})
        self._set_transition("on", {"the": 3.0, "a": 2.5})
        self._set_transition("mat", {"</s>": 3.0, "and": 1.5})
        self._set_transition("big", {"cat": 2.5, "dog": 2.5, "red": 1.5})
        self._set_transition("small", {"cat": 2.5, "dog": 2.5, "red": 1.0})
        self._set_transition("red", {"cat": 2.0, "dog": 2.0, "mat": 2.5})
        self._set_transition("ran", {"quickly": 2.5, "slowly": 2.0, "and": 1.0, "</s>": 1.5})
        self._set_transition("quickly", {"</s>": 2.5, "and": 1.5})
        self._set_transition("slowly", {"</s>": 2.5, "and": 1.5})
        self._set_transition("and", {"the": 2.5, "a": 2.0, "ran": 1.5, "sat": 1.5})
        self._set_transition("chased", {"the": 2.5, "a": 2.0})
    
    def _set_transition(self, from_token: str, transitions: Dict[str, float]):
        """Set logit values for transitions from a token."""
        from_id = self.token_to_id[from_token]
        for to_token, logit in transitions.items():
            to_id = self.token_to_id[to_token]
            self._logits[from_id, to_id] = logit
    
    def get_logits(self, token_id: int) -> np.ndarray:
        """Get next-token logits given current token."""
        return self._logits[token_id].copy()
    
    def get_probs(self, token_id: int, temperature: float = 1.0) -> np.ndarray:
        """Get next-token probabilities with optional temperature."""
        logits = self.get_logits(token_id) / temperature
        exp_logits = np.exp(logits - np.max(logits))  # numerical stability
        return exp_logits / exp_logits.sum()


model = SimpleLanguageModel()
print(f"Vocabulary ({model.vocab_size} tokens): {model.vocab}")
print()

# Show probabilities from <s>
probs = model.get_probs(model.token_to_id["<s>"])
print("P(next | <s>):")
sorted_indices = np.argsort(probs)[::-1]
for idx in sorted_indices[:5]:
    print(f"  {model.vocab[idx]:12s} {probs[idx]:.4f}")

In [ ]:
def greedy_decode(model: SimpleLanguageModel, max_length: int = 15) -> List[str]:
    """Greedy decoding: always pick the most likely next token."""
    tokens = ["<s>"]
    current_id = model.token_to_id["<s>"]
    
    for _ in range(max_length):
        probs = model.get_probs(current_id)
        next_id = np.argmax(probs)
        next_token = model.vocab[next_id]
        
        tokens.append(next_token)
        
        if next_token == "</s>":
            break
        
        current_id = next_id
    
    return tokens


# Run greedy decoding
greedy_output = greedy_decode(model)
print("Greedy Decoding:")
print(f"  Tokens: {greedy_output}")
print(f"  Text:   {' '.join(greedy_output[1:-1])}")
print()

# Run it multiple times -- always the same!
print("Greedy is deterministic (same output every time):")
for i in range(3):
    output = greedy_decode(model)
    print(f"  Run {i+1}: {' '.join(output[1:-1])}")

## 5. Beam Search

Beam search maintains $k$ (beam width) candidate sequences at each step and expands each by all possible next tokens, keeping only the top $k$ overall.

### Algorithm

1. Start with beam = `[("<s>", 0.0)]` (sequence, log-probability).
2. For each sequence in the beam, generate all possible next tokens.
3. Score each expanded sequence by cumulative log-probability.
4. Keep only the top $k$ sequences.
5. Repeat until all sequences end with `</s>` or max length.

### Trade-offs

| Beam Width $k$ | Quality | Diversity | Speed |
|:-:|:-:|:-:|:-:|
| 1 | Low (= greedy) | None | Fast |
| 3-5 | Good | Low | Medium |
| 10+ | Diminishing returns | Very low | Slow |

In [ ]:
def beam_search(model: SimpleLanguageModel, beam_width: int = 3,
                max_length: int = 15) -> List[Tuple[List[str], float]]:
    """Beam search decoding.
    
    Args:
        model: Language model.
        beam_width: Number of candidates to keep at each step.
        max_length: Maximum sequence length.
    
    Returns:
        List of (token_sequence, log_probability) tuples, sorted by probability.
    """
    # Each beam entry: (token_list, cumulative_log_prob)
    beams = [(["<s>"], 0.0)]
    completed = []
    
    for step in range(max_length):
        all_candidates = []
        
        for tokens, log_prob in beams:
            current_token = tokens[-1]
            
            # Skip completed sequences
            if current_token == "</s>":
                completed.append((tokens, log_prob))
                continue
            
            current_id = model.token_to_id[current_token]
            probs = model.get_probs(current_id)
            
            # Expand to all possible next tokens
            for next_id in range(model.vocab_size):
                next_token = model.vocab[next_id]
                next_log_prob = log_prob + np.log(probs[next_id] + 1e-10)
                all_candidates.append((tokens + [next_token], next_log_prob))
        
        if not all_candidates:
            break
        
        # Keep top-k candidates
        all_candidates.sort(key=lambda x: x[1], reverse=True)
        beams = all_candidates[:beam_width]
    
    # Add any remaining beams to completed
    completed.extend(beams)
    completed.sort(key=lambda x: x[1], reverse=True)
    
    return completed


# Run beam search with different widths
for width in [1, 3, 5]:
    results = beam_search(model, beam_width=width, max_length=10)
    best_tokens, best_log_prob = results[0]
    best_text = " ".join(best_tokens[1:])  # skip <s>
    if "</s>" in best_text:
        best_text = best_text[:best_text.index("</s>")].strip()
    
    print(f"Beam width={width}: {best_text:40s}  (log_prob={best_log_prob:.4f})")

print()
print("Beam width=1 is identical to greedy decoding.")
print("Wider beams may find different (sometimes better) sequences.")

In [ ]:
# ---- Show all beam search candidates ----

results = beam_search(model, beam_width=5, max_length=10)

print("Top 5 Beam Search Results:")
print("=" * 60)
for i, (tokens, log_prob) in enumerate(results[:5]):
    text = " ".join(tokens[1:])  # skip <s>
    if "</s>" in text:
        text = text[:text.index("</s>")].strip()
    print(f"  {i+1}. [{log_prob:>7.3f}] {text}")

## 6. Temperature Scaling

Temperature $T$ reshapes the probability distribution before sampling:

$$p_i = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$

where $z_i$ are the raw logits.

| Temperature | Effect | Use case |
|:-----------:|--------|----------|
| $T \to 0$ | Approaches greedy (argmax) | Most deterministic, factual |
| $T = 1$ | Original distribution | Default |
| $T > 1$ | Flatter distribution, more uniform | More creative/diverse |
| $T \to \infty$ | Uniform distribution | Maximum randomness |

In [ ]:
# ---- Demonstrate temperature effect ----

def softmax_with_temperature(logits: np.ndarray, temperature: float) -> np.ndarray:
    """Apply temperature-scaled softmax."""
    if temperature <= 0:
        # Approach argmax
        probs = np.zeros_like(logits)
        probs[np.argmax(logits)] = 1.0
        return probs
    
    scaled = logits / temperature
    exp_scaled = np.exp(scaled - np.max(scaled))  # numerical stability
    return exp_scaled / exp_scaled.sum()


# Get logits for "the" -> next token
the_id = model.token_to_id["the"]
logits = model.get_logits(the_id)

temperatures = [0.1, 0.5, 1.0, 2.0, 5.0]

print("Temperature Effect on P(next | 'the'):")
print("=" * 75)
print(f"{'Token':<12}", end="")
for t in temperatures:
    print(f"{'T='+str(t):>12}", end="")
print()
print("-" * 75)

# Show top tokens across all temperatures
base_probs = softmax_with_temperature(logits, 1.0)
top_indices = np.argsort(base_probs)[::-1][:8]

for idx in top_indices:
    print(f"{model.vocab[idx]:<12}", end="")
    for t in temperatures:
        prob = softmax_with_temperature(logits, t)[idx]
        print(f"{prob:>12.4f}", end="")
    print()

print()
print("Observations:")
print("  T=0.1: Almost all probability on top token (near-greedy).")
print("  T=1.0: Original distribution.")
print("  T=5.0: Nearly uniform -- even unlikely tokens get sampled.")

In [ ]:
# ---- Generate with different temperatures ----

def sample_decode(model: SimpleLanguageModel, temperature: float = 1.0,
                  max_length: int = 15, seed: int = 42) -> List[str]:
    """Decode by sampling from the temperature-scaled distribution."""
    rng = np.random.RandomState(seed)
    tokens = ["<s>"]
    current_id = model.token_to_id["<s>"]
    
    for _ in range(max_length):
        probs = model.get_probs(current_id, temperature=temperature)
        next_id = rng.choice(model.vocab_size, p=probs)
        next_token = model.vocab[next_id]
        
        tokens.append(next_token)
        if next_token == "</s>":
            break
        current_id = next_id
    
    return tokens


print("Generation with Different Temperatures:")
print("=" * 60)
for temp in [0.1, 0.5, 1.0, 2.0]:
    # Generate 3 samples at each temperature
    for seed in [42, 123, 456]:
        tokens = sample_decode(model, temperature=temp, seed=seed)
        text = " ".join(tokens[1:])
        if "</s>" in text:
            text = text[:text.index("</s>")].strip()
        print(f"  T={temp:<4}  seed={seed:<4} : {text}")
    print()

## 7. Top-k and Top-p Sampling (Preview)

These strategies restrict sampling to a subset of the vocabulary, reducing the chance of sampling very unlikely (and often incoherent) tokens.

### Top-k Sampling

Only sample from the **$k$ most probable** tokens. Set probability of all others to 0, then renormalize.

### Top-p (Nucleus) Sampling

Only sample from the **smallest set of tokens** whose cumulative probability exceeds $p$. This adaptively adjusts the number of candidates.

### Comparison

| Aspect | Top-k | Top-p |
|--------|:-----:|:-----:|
| Fixed # candidates | Yes ($k$) | No (varies) |
| Adaptive | No | Yes |
| Problem | $k$ too large for peaked dist, too small for flat | Threshold $p$ needs tuning |
| Typical values | $k = 40$ to $100$ | $p = 0.9$ to $0.95$ |

In [ ]:
def top_k_filter(probs: np.ndarray, k: int) -> np.ndarray:
    """Zero out all but the top-k probabilities, then renormalize."""
    if k >= len(probs):
        return probs.copy()
    
    filtered = np.zeros_like(probs)
    top_k_indices = np.argsort(probs)[-k:]
    filtered[top_k_indices] = probs[top_k_indices]
    
    # Renormalize
    total = filtered.sum()
    if total > 0:
        filtered /= total
    return filtered


def top_p_filter(probs: np.ndarray, p: float) -> np.ndarray:
    """Zero out tokens outside the nucleus (top-p), then renormalize."""
    sorted_indices = np.argsort(probs)[::-1]
    sorted_probs = probs[sorted_indices]
    cumulative = np.cumsum(sorted_probs)
    
    # Find cutoff: smallest set where cumulative >= p
    cutoff_idx = np.searchsorted(cumulative, p) + 1
    
    filtered = np.zeros_like(probs)
    keep_indices = sorted_indices[:cutoff_idx]
    filtered[keep_indices] = probs[keep_indices]
    
    # Renormalize
    total = filtered.sum()
    if total > 0:
        filtered /= total
    return filtered


# Demonstrate on P(next | "the")
probs = model.get_probs(the_id)

print("Original distribution P(next | 'the'):")
sorted_idx = np.argsort(probs)[::-1]
for idx in sorted_idx[:8]:
    print(f"  {model.vocab[idx]:<12} {probs[idx]:.4f}")

print()

# Top-k
for k in [3, 5]:
    filtered = top_k_filter(probs, k)
    non_zero = [(model.vocab[i], filtered[i]) for i in range(len(filtered)) if filtered[i] > 0]
    print(f"Top-k (k={k}): {non_zero}")

print()

# Top-p
for p in [0.8, 0.95]:
    filtered = top_p_filter(probs, p)
    non_zero = [(model.vocab[i], round(filtered[i], 4)) for i in range(len(filtered)) if filtered[i] > 0]
    print(f"Top-p (p={p}): {non_zero}")

## 8. Output Filtering

After decoding, we may need to **filter** the output for quality:

- **Repetition removal**: Remove repeated phrases or sentences.
- **Length constraints**: Enforce minimum/maximum output length.
- **Safety checks**: Flag or remove potentially harmful content (conceptual).

In [ ]:
# ---- Repetition detection and removal ----

def detect_ngram_repetition(tokens: List[str], n: int = 3) -> List[Tuple[int, tuple]]:
    """Detect repeated n-grams in token sequence."""
    seen = {}
    repetitions = []
    
    for i in range(len(tokens) - n + 1):
        ngram = tuple(tokens[i:i+n])
        if ngram in seen:
            repetitions.append((i, ngram))
        else:
            seen[ngram] = i
    
    return repetitions


def remove_repeated_sentences(text: str) -> str:
    """Remove consecutive duplicate sentences."""
    import re
    sentences = re.split(r'(?<=[.!?])\s+', text)
    filtered = [sentences[0]] if sentences else []
    
    for sent in sentences[1:]:
        if sent != filtered[-1]:
            filtered.append(sent)
    
    return " ".join(filtered)


# Example: repetitive output
repetitive_text = (
    "The cat sat on the mat. The cat sat on the mat. "
    "The dog ran quickly. The dog ran quickly. "
    "It was a good day."
)

print("Repetition Removal:")
print(f"  Before: {repetitive_text}")
print(f"  After:  {remove_repeated_sentences(repetitive_text)}")
print()

# N-gram repetition detection
tokens = "the cat sat on the mat the cat sat on the rug".split()
reps = detect_ngram_repetition(tokens, n=3)
print(f"3-gram repetitions in '{' '.join(tokens)}':")
for pos, ngram in reps:
    print(f"  Position {pos}: {ngram}")

In [ ]:
# ---- Length constraint enforcement ----

def enforce_length(text: str, min_words: int = 5, max_words: int = 50) -> str:
    """Enforce word-count length constraints on generated text."""
    words = text.split()
    
    if len(words) < min_words:
        return None  # signal that output is too short
    
    if len(words) > max_words:
        # Truncate at the last sentence boundary before max_words
        truncated = " ".join(words[:max_words])
        # Find last sentence-ending punctuation
        import re
        match = list(re.finditer(r'[.!?]', truncated))
        if match:
            return truncated[:match[-1].end()]
        return truncated + "..."
    
    return text


# Examples
short_text = "Hello world."
long_text = ("The transformer architecture has revolutionized natural language processing. "
             "It uses self-attention mechanisms to process entire sequences in parallel. "
             "This approach has led to significant improvements in translation, summarization, "
             "and question answering tasks. The key innovation is the multi-head attention "
             "mechanism that allows the model to attend to different parts of the input "
             "simultaneously. This has made transformers the dominant architecture in modern NLP.")

print("Length Constraint Enforcement:")
print(f"  Short ({len(short_text.split())} words): {enforce_length(short_text, min_words=5)}")
print(f"  Long  ({len(long_text.split())} words), max_words=20:")
print(f"    {enforce_length(long_text, max_words=20)}")

In [ ]:
# ---- Conceptual safety filtering ----

def simple_content_filter(text: str, blocked_patterns: List[str] = None
                          ) -> Tuple[str, bool]:
    """Basic content filter (conceptual demonstration only).
    
    In production, use dedicated content moderation APIs/models.
    This is a simplified illustration of the concept.
    
    Returns:
        (filtered_text, was_filtered)
    """
    import re
    
    if blocked_patterns is None:
        # Example patterns (benign for demonstration)
        blocked_patterns = [r"\bTODO\b", r"\bFIXME\b", r"\bHACK\b"]
    
    was_filtered = False
    filtered = text
    for pattern in blocked_patterns:
        if re.search(pattern, filtered, re.IGNORECASE):
            filtered = re.sub(pattern, "[FILTERED]", filtered, flags=re.IGNORECASE)
            was_filtered = True
    
    return filtered, was_filtered


# Demonstration
test_texts = [
    "This is a normal response about language models.",
    "TODO: Fix this HACK in the codebase FIXME.",
    "The results look good, no issues found.",
]

print("Content Filtering (Conceptual):")
print("=" * 60)
for text in test_texts:
    filtered, was_flagged = simple_content_filter(text)
    flag = " [FLAGGED]" if was_flagged else ""
    print(f"  Input:  {text}")
    print(f"  Output: {filtered}{flag}")
    print()

print("Note: Real safety systems use ML-based classifiers, not pattern matching.")

## 9. Common Mistakes & Debugging Tips

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Forgetting to remove special tokens | `[CLS]` and `[SEP]` in output text | Always strip special tokens during detokenization |
| Using greedy for creative tasks | Repetitive, boring output | Use sampling with temperature or top-p |
| Temperature too high | Incoherent, random output | Start with $T=0.7$-$1.0$ and adjust |
| Temperature too low | Nearly greedy, no diversity | Increase $T$ or use top-p |
| Beam search too wide | Very slow, minimal quality gain | $k=3$-$5$ is usually sufficient |
| Not normalizing beam scores by length | Shorter sequences always win | Divide log-prob by sequence length |
| Wrong subword merging scheme | Mangled output text | Match detokenizer to the tokenizer used |
| No repetition penalty | Model repeats phrases endlessly | Add n-gram repetition penalty or post-filter |

**Debugging tip:** If your generated text looks garbled, check: (1) Are you using the right detokenizer? (2) Are special tokens being removed? (3) Is the vocabulary mapping correct?

## 10. Exercises

### Exercise 1: Implement Top-k Sampling Decoder

Combine top-k filtering with sampling to build a full decoder. Generate text with different values of $k$ and compare the results.

In [ ]:
# TODO:
# 1. Write a function top_k_sample_decode(model, k, temperature, max_length, seed)
#    that generates text using top-k filtered sampling.
# 2. Generate 3 samples each with k=2, k=5, k=10.
# 3. Compare the diversity and quality of outputs.
# Your code here

### Exercise 2: Beam Search with Length Normalization

Standard beam search favors shorter sequences (fewer tokens = fewer negative log-prob terms). Implement length-normalized beam search.

In [ ]:
# TODO:
# 1. Modify the beam_search function to normalize scores by sequence length.
#    Use: normalized_score = log_prob / (len(tokens) ** alpha)
#    where alpha is a length penalty parameter (try alpha=0.6).
# 2. Compare results with and without length normalization.
# Your code here

## 11. Exercise Solutions

In [ ]:
# ---- Exercise 1 Solution ----

def top_k_sample_decode(model: SimpleLanguageModel, k: int = 5,
                        temperature: float = 1.0, max_length: int = 15,
                        seed: int = 42) -> List[str]:
    """Decode using top-k sampling."""
    rng = np.random.RandomState(seed)
    tokens = ["<s>"]
    current_id = model.token_to_id["<s>"]
    
    for _ in range(max_length):
        probs = model.get_probs(current_id, temperature=temperature)
        filtered_probs = top_k_filter(probs, k)
        next_id = rng.choice(model.vocab_size, p=filtered_probs)
        next_token = model.vocab[next_id]
        
        tokens.append(next_token)
        if next_token == "</s>":
            break
        current_id = next_id
    
    return tokens


print("Exercise 1: Top-k Sampling")
print("=" * 60)
for k in [2, 5, 10]:
    print(f"\nk={k}:")
    for seed in [42, 123, 456]:
        tokens = top_k_sample_decode(model, k=k, temperature=1.0, seed=seed)
        text = " ".join(tokens[1:])
        if "</s>" in text:
            text = text[:text.index("</s>")].strip()
        print(f"  seed={seed:<4} : {text}")

print()
print("Observation: Smaller k -> more focused; larger k -> more diverse.")

In [ ]:
# ---- Exercise 2 Solution ----

def beam_search_normalized(model: SimpleLanguageModel, beam_width: int = 3,
                           max_length: int = 15, alpha: float = 0.6
                           ) -> List[Tuple[List[str], float]]:
    """Beam search with length normalization."""
    beams = [(["<s>"], 0.0)]
    completed = []
    
    for step in range(max_length):
        all_candidates = []
        
        for tokens, log_prob in beams:
            current_token = tokens[-1]
            
            if current_token == "</s>":
                completed.append((tokens, log_prob))
                continue
            
            current_id = model.token_to_id[current_token]
            probs = model.get_probs(current_id)
            
            for next_id in range(model.vocab_size):
                next_token = model.vocab[next_id]
                next_log_prob = log_prob + np.log(probs[next_id] + 1e-10)
                all_candidates.append((tokens + [next_token], next_log_prob))
        
        if not all_candidates:
            break
        
        # Sort by LENGTH-NORMALIZED score
        all_candidates.sort(
            key=lambda x: x[1] / (len(x[0]) ** alpha),
            reverse=True
        )
        beams = all_candidates[:beam_width]
    
    completed.extend(beams)
    completed.sort(key=lambda x: x[1] / (len(x[0]) ** alpha), reverse=True)
    return completed


print("Exercise 2: Length-Normalized vs Standard Beam Search")
print("=" * 65)

for name, search_fn in [("Standard", beam_search),
                         ("Length-normalized", beam_search_normalized)]:
    results = search_fn(model, beam_width=5, max_length=10)
    best_tokens, best_log_prob = results[0]
    text = " ".join(best_tokens[1:])
    if "</s>" in text:
        text = text[:text.index("</s>")].strip()
    norm_score = best_log_prob / (len(best_tokens) ** 0.6)
    print(f"  {name:<20s}: {text}")
    print(f"    log_prob={best_log_prob:.3f}, len={len(best_tokens)}, "
          f"norm_score={norm_score:.3f}")
    print()

print("Length normalization prevents short sequences from dominating.")